# Jour 4 — Évaluation & Analyse des runs

Ce notebook :
1. Charge et visualise les courbes d'apprentissage (Jour 3)
2. Évalue le checkpoint final en mode déterministe
3. Analyse les résultats du sweep hyperparamètres (si disponible)

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## 1. Courbes d'apprentissage — Jour 3

In [ ]:
RUN_DIR = '../runs/day3_run'

ep_df  = pd.read_csv(os.path.join(RUN_DIR, 'episodes.csv'))
upd_df = pd.read_csv(os.path.join(RUN_DIR, 'updates.csv'))

print(f'Épisodes  : {len(ep_df)}')
print(f'Updates   : {len(upd_df)}')
ep_df.tail()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Return lissé
window = max(1, len(ep_df) // 20)
ep_df['return_smooth'] = ep_df['ep_return'].rolling(window, min_periods=1).mean()

axes[0].plot(ep_df['total_steps'], ep_df['ep_return'], alpha=0.3, color='tab:blue')
axes[0].plot(ep_df['total_steps'], ep_df['return_smooth'], color='tab:blue', linewidth=2)
axes[0].set_title('Return par épisode')
axes[0].set_xlabel('Steps')

# % temps en zone sûre
if 'pct_in_safe' in ep_df.columns:
    ep_df['pct_smooth'] = ep_df['pct_in_safe'].rolling(window, min_periods=1).mean()
    axes[1].plot(ep_df['total_steps'], ep_df['pct_in_safe'], alpha=0.3, color='tab:green')
    axes[1].plot(ep_df['total_steps'], ep_df['pct_smooth'], color='tab:green', linewidth=2)
    axes[1].set_title('% steps en zone sûre')
    axes[1].set_xlabel('Steps')
    axes[1].set_ylim(0, 105)

# T_max
if 'T_max' in ep_df.columns:
    axes[2].plot(ep_df['total_steps'], ep_df['T_max'], alpha=0.3, color='tab:red')
    axes[2].plot(ep_df['total_steps'],
                 ep_df['T_max'].rolling(window, min_periods=1).mean(),
                 color='tab:red', linewidth=2)
    axes[2].axhline(45, color='orange', linestyle='--', label='T_safe_max')
    axes[2].axhline(60, color='red',    linestyle=':',  label='T_cutoff')
    axes[2].set_title('T_max par épisode')
    axes[2].set_xlabel('Steps')
    axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(RUN_DIR, 'learning_curves.png'), dpi=120)
plt.show()
print('Sauvegardé → learning_curves.png')

In [ ]:
# Losses SAC
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, title, color in [
    (axes[0], 'critic_loss', 'Critic Loss',  'tab:blue'),
    (axes[1], 'actor_loss',  'Actor Loss',   'tab:orange'),
    (axes[2], 'alpha',       'Alpha (entropie)', 'tab:green'),
]:
    if col in upd_df.columns:
        w = max(1, len(upd_df) // 50)
        ax.plot(upd_df['step'], upd_df[col], alpha=0.2, color=color)
        ax.plot(upd_df['step'], upd_df[col].rolling(w, min_periods=1).mean(),
                color=color, linewidth=2)
        ax.set_title(title)
        ax.set_xlabel('Step')

plt.tight_layout()
plt.show()

## 2. Évaluation du checkpoint final

In [ ]:
from config import EnvConfig, ThermalConfig, RewardConfig
from envs.battery_thermal_env import BatteryThermalEnv
from evaluate import load_agent, run_episode, plot_episode

CHECKPOINT = '../runs/day3_run/checkpoints/sac_final.pt'
N_EVAL_EPISODES = 10

env_cfg = EnvConfig(thermal=ThermalConfig(), reward=RewardConfig())
env     = BatteryThermalEnv(config=env_cfg)

agent = load_agent(
    CHECKPOINT,
    obs_dim    = env.observation_space.shape[0],
    action_dim = env.action_space.shape[0],
    hidden_dim = 256,
    n_layers   = 2,
)
print('Agent chargé.')

In [ ]:
results = [run_episode(agent, env, seed=i) for i in range(N_EVAL_EPISODES)]

df_eval = pd.DataFrame([{
    'ep':        i+1,
    'return':    r['return'],
    'pct_safe':  r['pct_safe'],
    'T_max':     r['T_max'],
    'T_mean':    r['T_mean'],
    'SoC_final': r['SoC_final'],
    'length':    r['length'],
} for i, r in enumerate(results)])

print(df_eval.to_string(index=False))
print(f"\nMoyenne return   : {df_eval['return'].mean():.2f} ± {df_eval['return'].std():.2f}")
print(f"Moyenne pct_safe : {df_eval['pct_safe'].mean():.1f}%")
print(f"Moyenne T_max    : {df_eval['T_max'].mean():.1f}°C")

In [ ]:
# Trajectoire du meilleur épisode
best_idx = int(df_eval['return'].idxmax())
best_ep  = results[best_idx]
tc       = env.tc
steps    = range(len(best_ep['T_hist']))

fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=True)

# Température
ax = axes[0]
ax.plot(steps, best_ep['T_hist'], color='tab:red', linewidth=1.2, label='T_cell')
ax.axhline(tc.T_safe_min, color='blue',   linestyle='--', alpha=0.7, label=f'T_safe_min ({tc.T_safe_min}°C)')
ax.axhline(tc.T_safe_max, color='orange', linestyle='--', alpha=0.7, label=f'T_safe_max ({tc.T_safe_max}°C)')
ax.axhline(tc.T_cutoff,   color='red',    linestyle=':',  alpha=0.7, label=f'T_cutoff ({tc.T_cutoff}°C)')
ax.fill_between(steps, tc.T_safe_min, tc.T_safe_max, alpha=0.08, color='green', label='Zone sûre')
ax.set_ylabel('Température (°C)')
ax.legend(fontsize=8)

# SoC
ax = axes[1]
ax.plot(steps, best_ep['SoC_hist'], color='tab:green', linewidth=1.2)
ax.set_ylabel('SoC')
ax.set_ylim(0, 1.05)

# Action
ax = axes[2]
ax.fill_between(steps, 0, best_ep['action_hist'], color='tab:blue', alpha=0.6)
ax.plot(steps, best_ep['action_hist'], color='tab:blue', linewidth=0.8)
ax.set_ylabel('Refroidissement (0–1)')
ax.set_xlabel('Step')
ax.set_ylim(-0.02, 1.02)

fig.suptitle(
    f"Meilleur épisode (ep {best_idx+1}) — "
    f"return={best_ep['return']:.1f}  "
    f"pct_safe={best_ep['pct_safe']:.1f}%  "
    f"T_max={best_ep['T_max']:.1f}°C",
    fontsize=11
)
plt.tight_layout()
plt.show()

## 3. Analyse du sweep (si disponible)

Lance d'abord : `python sweep.py --total_steps 5000`

In [ ]:
import glob

sweep_files = sorted(glob.glob('../runs/sweep_*/sweep_results.csv'))
if not sweep_files:
    print('Aucun résultat de sweep trouvé.')
    print('Lance : python sweep.py --total_steps 5000')
else:
    sweep_path = sweep_files[-1]  # dernier sweep
    print(f'Chargement : {sweep_path}')
    sw = pd.read_csv(sweep_path)
    sw = sw.sort_values('eval_return_mean', ascending=False)
    print(sw.to_string(index=False))

In [ ]:
if sweep_files:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Return par lr et gamma
    for gamma_val in sw['gamma'].unique():
        sub = sw[sw['gamma'] == gamma_val].sort_values('lr')
        axes[0].plot(sub['lr'].astype(str), sub['eval_return_mean'],
                     marker='o', label=f'gamma={gamma_val}')
    axes[0].set_title('Return moyen vs Learning Rate')
    axes[0].set_xlabel('Learning Rate')
    axes[0].legend()

    # % safe par tau
    for tau_val in sw['tau'].unique():
        sub = sw[sw['tau'] == tau_val].sort_values('lr')
        axes[1].plot(sub['lr'].astype(str), sub['eval_pct_safe'],
                     marker='s', label=f'tau={tau_val}')
    axes[1].set_title('% Safe vs Learning Rate')
    axes[1].set_xlabel('Learning Rate')
    axes[1].legend()

    plt.tight_layout()
    plt.show()